# **Ejercicio 1 - Preparación del ambiente**

Este notebook valida que el ambiente levantado con Docker Compose funciona y deja
evidencia de las herramientas disponibles.

## **Arquitectura del ambiente**

`docker compose up --build` levanta dos servicios (dos contenedores):

| Servicio | Contenedor | Puerto | Qué contiene |
|---|---|---|---|
| `lab` | `lab8-lab` | 8888 | JupyterLab + Python. DuckDB vive aquí como librería (`import duckdb`), no como un servidor aparte. |
| `metabase` | `lab8-metabase` | 3000 | Herramienta de tableros (BI) con el driver de DuckDB. Solo visualiza; no es la fuente de los datos. |

**1.1 y 1.2:** el fork del repositorio del docente y el procedimiento para levantar el
ambiente están documentados en el `README.md`, sección *Como levantar el ambiente* (1.5).

## **1.3 Verificación de los servicios**

### **1.3.a Servicio `lab`: el kernel corre dentro del contenedor**

In [1]:
import os
import platform
import sys
from pathlib import Path

print(f"Python      : {sys.version.split()[0]}")
print(f"Sistema     : {platform.system()} ({platform.platform()})")
print(f"Hostname    : {platform.node()}")
print(f"Directorio  : {Path.cwd()}")
print(f"/workspace  : {'existe' if Path('/workspace').is_dir() else 'NO existe'}")

Python      : 3.11.14
Sistema     : Linux (Linux-6.8.0-50-generic-aarch64-with-glibc2.41)
Hostname    : d27bb830fec0
Directorio  : /workspace/notebooks
/workspace  : existe


### **1.3.b DuckDB funciona: consulta SQL y ciclo completo con Parquet**

Primero se hace una consulta sobre datos generados por el propio DuckDB, y luego se escribe
un archivo Parquet de prueba para consultarlo directamente, que es el patrón que se usará
en todo el laboratorio.

In [2]:
import duckdb

con = duckdb.connect()  # base en memoria: no deja archivos

con.sql("""
    SELECT version()                AS version_duckdb,
           COUNT(*)                 AS filas_generadas,
           SUM(n)::BIGINT           AS suma,
           ROUND(AVG(n), 1)         AS promedio
    FROM range(1, 1000001) AS t(n)
""").df()

,version_duckdb,filas_generadas,suma,promedio
0,v1.5.5,1000000,500000500000,500000.5


In [3]:
import tempfile

dir_prueba = Path(tempfile.mkdtemp())
ruta_parquet = dir_prueba / "prueba.parquet"

# Escritura: 10,000 "viajes" sintéticos con tipo de taxi y tarifa
con.sql(f"""
    COPY (
        SELECT n                                        AS id_viaje,
               CASE WHEN n % 4 = 0 THEN 'green' ELSE 'yellow' END AS tipo_taxi,
               ROUND(5 + (n % 50) * 0.75, 2)            AS tarifa
        FROM range(10000) AS t(n)
    ) TO '{ruta_parquet}' (FORMAT parquet)
""")
print(f"[escrito] {ruta_parquet} ({ruta_parquet.stat().st_size:,} bytes)")

# Lectura: consulta directa sobre el archivo, sin importarlo a una tabla
con.sql(f"""
    SELECT tipo_taxi,
           COUNT(*)              AS viajes,
           ROUND(AVG(tarifa), 2) AS tarifa_promedio
    FROM '{ruta_parquet}'
    GROUP BY tipo_taxi
    ORDER BY tipo_taxi
""").df()

[escrito] /tmp/tmpkysx6avp/prueba.parquet (49,927 bytes)


,tipo_taxi,viajes,tarifa_promedio
0,green,2500,23.0
1,yellow,7500,23.5


In [4]:
import shutil

shutil.rmtree(dir_prueba)
con.close()
print(f"[limpieza] eliminado {dir_prueba}")

[limpieza] eliminado /tmp/tmpkysx6avp


### **1.3.c Montaje de la carpeta `data/`**

Los datos se descargan en la carpeta del proyecto y deben verse desde el contenedor, por lo
que se verifica que existan los directorios y que se pueda escribir en ellos.

In [5]:
DIR_DATA = Path("/workspace/data")

for sub in ("raw", "processed"):
    directorio = DIR_DATA / sub
    marca = directorio / ".prueba_escritura"
    marca.write_text("ok")
    escribible = marca.read_text() == "ok"
    marca.unlink()
    print(f"{directorio}: existe={directorio.is_dir()} escribible={escribible}")

/workspace/data/raw: existe=True escribible=True
/workspace/data/processed: existe=True escribible=True


### **1.3.d Servicio `metabase`**

Dentro de la red que crea Docker Compose, los contenedores se encuentran por el nombre del
servicio. Por eso, desde `lab` se consulta `http://metabase:3000` y no `localhost`, ya que
`localhost` dentro del contenedor es el propio contenedor `lab`. Desde la computadora,
Metabase se abre en <http://localhost:3000>.

In [6]:
import requests

respuesta = requests.get("http://metabase:3000/api/health", timeout=10)
print(f"HTTP {respuesta.status_code} -> {respuesta.json()}")
assert respuesta.ok and respuesta.json().get("status") == "ok", "Metabase no está disponible"
print("[ok] Metabase responde")

HTTP 200 -> {'status': 'ok'}
[ok] Metabase responde


## **1.4 Herramientas disponibles en el ambiente**

Las versiones esperadas se leen de `requirements.txt`, la misma fuente que usa el
`Dockerfile`, de tal forma que la verificación no depende de valores escritos a mano.

| Herramienta | Contenedor | Uso en el laboratorio |
|---|---|---|
| Python 3.11 | `lab` | Lenguaje base de scripts y notebooks |
| duckdb | `lab` | Motor de consultas SQL sobre Parquet y tablas |
| jupyterlab | `lab` | Entorno de notebooks |
| pandas | `lab` | Recibir resultados de DuckDB como DataFrame (`.df()`) |
| pyarrow | `lab` | Lectura y escritura de Parquet, puente entre DuckDB y pandas |
| matplotlib | `lab` | Gráficas dentro de los notebooks |
| requests | `lab` | Descargas HTTP del script `download_data.py` |
| Metabase + driver DuckDB | `metabase` | Tablero del Ejercicio 7 |

In [7]:
from importlib.metadata import PackageNotFoundError, version

import pandas as pd

requisitos = {}
for linea in Path("/workspace/requirements.txt").read_text().splitlines():
    linea = linea.split("#")[0].strip()
    if "==" in linea:
        paquete, esperada = linea.split("==")
        requisitos[paquete.strip()] = esperada.strip()

filas = []
for paquete, esperada in requisitos.items():
    try:
        instalada = version(paquete)
    except PackageNotFoundError:
        instalada = None
    filas.append({
        "paquete": paquete,
        "version_esperada": esperada,
        "version_instalada": instalada or "NO INSTALADO",
        "estado": "OK" if instalada == esperada else "FALLO",
    })

inventario = pd.DataFrame(filas)
inventario

,paquete,version_esperada,version_instalada,estado
0,duckdb,1.5.5,1.5.5,OK
1,jupyterlab,4.6.4,4.6.4,OK
2,pandas,3.0.6,3.0.6,OK
3,pyarrow,25.0.1,25.0.1,OK
4,matplotlib,3.11.2,3.11.2,OK
5,requests,2.34.2,2.34.2,OK


In [8]:
assert (inventario["estado"] == "OK").all(), "Hay paquetes faltantes o con otra versión"

# duckdb debe coincidir con el driver instalado en Metabase (metabase.Dockerfile)
DRIVER_METABASE = "1.5.5.0"
assert DRIVER_METABASE.startswith(duckdb.__version__), (
    f"duckdb {duckdb.__version__} no coincide con el driver de Metabase {DRIVER_METABASE}"
)
print(f"[ok] {len(inventario)} paquetes con la versión esperada")
print(f"[ok] duckdb {duckdb.__version__} alineado con el driver de Metabase {DRIVER_METABASE}")

[ok] 6 paquetes con la versión esperada
[ok] duckdb 1.5.5 alineado con el driver de Metabase 1.5.5.0


## **1.5 Procedimiento para levantar el ambiente**

Documentado en el `README.md`, sección *Como levantar el ambiente*.

## **1.6 Importancia de un ambiente reproducible**

Considero que trabajar en un ambiente reproducible es importante porque garantiza que todos
los miembros del equipo ejecuten el proyecto bajo las mismas condiciones, sin importar el
sistema operativo o lo que cada uno tenga instalado en su computadora. Al definir el ambiente
con Docker, las versiones de Python, DuckDB, pandas y Metabase quedan fijas, de tal forma que
se evita el típico "en mi máquina sí funciona". Por ejemplo, en este laboratorio la versión
de DuckDB (1.5.5) debe coincidir con la del driver que utiliza Metabase (1.5.5.0), ya que una
diferencia entre ambas podría impedir que el tablero lea la base de datos.

Adicional, la reproducibilidad no solo beneficia al equipo durante el desarrollo, sino
también a cualquier persona que quiera verificar o retomar el análisis. Cabe mencionar que
Docker garantiza que el software sea el mismo, pero no los datos; eso lo cubren los scripts
de descarga, que obtienen la información siempre desde la fuente oficial de la TLC. Es por
esto que ambos, en conjunto, permiten reproducir los resultados.